# Training and Evaluation Notebook

This notebook evaluates zero-shot news classification for Indian headlines using the Kaggle India Headlines dataset and includes an ablation study.

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import pipeline

/mnt/Shared/SMAI_A3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download Dataset

Option A: Download manually from Kaggle and place CSV in `data/`.

Option B: Use kagglehub in Python (requires Kaggle credentials configured).

In [2]:
# Uncomment this block if using kagglehub
#import kagglehub
#path = kagglehub.dataset_download('therohk/india-headlines-news-dataset')
#print('Dataset downloaded to:', path)

In [3]:
DATA_PATH = Path('../data/filtered_data1.csv')
assert DATA_PATH.exists(), f'Missing dataset file at {DATA_PATH.resolve()}'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(5471, 3)


,publish_date,headline_category,headline_text
0,20100716,Sports,India beat Belgium 3-2 in 2nd hockey Test; lea...
1,20081202,Sports,Dhoni retains ODI top spot
2,20040503,Politics,Priyanka Gandhi as the Gen Now pick
3,20100311,Technology,Online CAT to continue despite glitches!
4,20030313,Entertainment,Comedy film strikes sour note with Sikhs


## 2. Prepare Labels

Adjust the source column names to match your CSV. Expected text column: `headline_text`.
Expected label column: `category`.

In [4]:
# Auto-detect common column names across dataset versions
text_candidates = ['headline_text', 'title', 'headline', 'news_title']
label_candidates = ['category', 'headline_category', 'section', 'topic']

TEXT_COL = next((c for c in text_candidates if c in df.columns), None)
LABEL_COL = next((c for c in label_candidates if c in df.columns), None)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        'Could not detect required columns. '
        f'Found columns: {list(df.columns)}. '
        f'Tried text columns: {text_candidates}, label columns: {label_candidates}'
    )

print('Using TEXT_COL =', TEXT_COL, '| LABEL_COL =', LABEL_COL)

labels_path = Path("../labels.txt")
if not labels_path.exists():
    raise FileNotFoundError(f"Missing labels file at {labels_path.resolve()}")

candidate_labels = [
    line.strip()
    for line in labels_path.read_text().splitlines()
    if line.strip()
 ]

if not candidate_labels:
    raise ValueError(f"No labels found in {labels_path.resolve()}")

print("Loaded candidate labels:", candidate_labels)

work = df[[TEXT_COL, LABEL_COL]].dropna().copy()
work[TEXT_COL] = work[TEXT_COL].astype(str).str.strip()
work[LABEL_COL] = work[LABEL_COL].astype(str).str.strip()
work = work[work[TEXT_COL] != '']

# Keep only rows whose labels match labels.txt
work = work[work[LABEL_COL].isin(candidate_labels)].copy()
work['x'] = work[TEXT_COL]
work['y'] = work[LABEL_COL]

print(work.shape)
work['y'].value_counts()

Using TEXT_COL = headline_text | LABEL_COL = headline_category
Loaded candidate labels: ['Business', 'Crime', 'Entertainment', 'Politics', 'Sports', 'Technology']
(5471, 4)


y
Sports           1000
Technology       1000
Entertainment    1000
Crime            1000
Business         1000
Politics          471
Name: count, dtype: int64

In [5]:
train_df, test_df = train_test_split(
    work[['x', 'y']],
    test_size=0.2,
    random_state=42,
    stratify=work['y']
)

print('Train:', train_df.shape, 'Test:', test_df.shape)

Train: (4376, 2) Test: (1095, 2)


## 3. Zero-shot Evaluation

In [6]:
labels_path = Path("../labels.txt")
if not labels_path.exists():
    raise FileNotFoundError(f"Missing labels file at {labels_path.resolve()}")

candidate_labels = [
    line.strip()
    for line in labels_path.read_text().splitlines()
    if line.strip()
]

if not candidate_labels:
    raise ValueError(f"No labels found in {labels_path.resolve()}")

print("Loaded candidate labels:", candidate_labels)

clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sample_test = test_df.sample(n=min(500, len(test_df)), random_state=42).reset_index(drop=True)
texts = sample_test["x"].tolist()
pred_obj = clf(
    texts,
    candidate_labels=candidate_labels,
    multi_label=False,
#    hypothesis_template="This news article is about {}.",
    hypothesis_template="This headline is about {}.",
)

pred_labels = [obj["labels"][0] for obj in pred_obj]
true_labels = sample_test["y"].tolist()

print(classification_report(true_labels, pred_labels, digits=4))

Loaded candidate labels: ['Business', 'Crime', 'Entertainment', 'Politics', 'Sports', 'Technology']


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2102.59it/s]


               precision    recall  f1-score   support

     Business     0.6147    0.7283    0.6667        92
        Crime     0.6635    0.8023    0.7263        86
Entertainment     0.7674    0.7174    0.7416        92
     Politics     0.4359    0.4146    0.4250        41
       Sports     0.8649    0.7442    0.8000        86
   Technology     0.7841    0.6699    0.7225       103

     accuracy                         0.7040       500
    macro avg     0.6884    0.6795    0.6803       500
 weighted avg     0.7144    0.7040    0.7053       500



In [7]:
cm = confusion_matrix(true_labels, pred_labels, labels=candidate_labels)
cm_df = pd.DataFrame(cm, index=candidate_labels, columns=candidate_labels)
cm_df

,Business,Crime,Entertainment,Politics,Sports,Technology
Business,67,3,2,6,5,9
Crime,4,69,1,9,1,2
Entertainment,4,15,66,2,2,3
Politics,2,9,9,17,1,3
Sports,4,6,6,4,64,2
Technology,28,2,2,1,1,69


## 4. Ablation Study

Compare two hypothesis templates to study prompt sensitivity in zero-shot classification.

In [ ]:
# ── Prompt-Tuning Ablation Study ────────────────────────────────────────────
#
# HOW BART-LARGE-MNLI WORKS:
#   Scores P(entailment | premise=headline, hypothesis=template.format(label))
#   Better templates create stronger NLI entailment signals per class.
#
# DESIGN PRINCIPLES USED:
#   1. Domain anchor  : "Indian" or "news" sets context
#   2. Strong verb    : "covers"/"focuses on"/"reports on" > "is about"
#   3. Category names : expanded names reduce inter-class confusion
#   4. Categorical    : "belongs to the {} category" aligns with MNLI style

templates = {
    # ── Baselines (already evaluated) ──────────────────────────────────────
    "P0_baseline_article":  "This news article is about {}.",
    "P1_baseline_indian":   "The primary topic of this Indian headline is {}.",
    "P2_baseline_headline": "This headline is about {}.",

    # ── New prompt-tuned candidates ────────────────────────────────────────
    # P3: "covers" is a stronger NLI entailment verb than "is about"
    "P3_covers":            "This Indian news headline covers {}.",

    # P4: categorical framing matches MNLI training distribution
    "P4_category":          "This headline belongs to the {} category.",

    # P5: journalism beat framing — headline reports on a topic
    "P5_reports_on":        "This Indian headline reports on {}.",

    # P6: newsroom section taxonomy BART has likely seen in pre-training
    "P6_section":           "This article is filed under the {} section.",

    # P7: "focuses on" is a high-precision entailment phrase in NLI corpora
    "P7_focuses":           "This Indian news headline focuses on {}.",

    # P8: subject-noun form — clean NLI hypothesis; uses EXPANDED labels
    "P8_subject_expanded":  "The subject of this Indian headline is {}.",

    # P9: most specific — domain anchor + verb + "topic of"; EXPANDED labels
    "P9_specific_expanded": "This Indian news headline focuses on the topic of {}.",
}

# Expanded candidate label names for P8 and P9.
# The confusion matrix shows Technology absorbs Business/Politics/Entertainment
# headlines. Richer label strings anchor the NLI hypothesis more precisely.
expanded_labels_map = {
    "Politics":      "politics and government",
    "Sports":        "sports and games",
    "Technology":    "science and technology",
    "Business":      "business and economy",
    "Entertainment": "entertainment and cinema",
    "Crime":         "crime and law",
    "Science":       "science and research",
    "Finance":       "finance and banking",
}

ablation_rows = []

for key, template in templates.items():
    # Use expanded label names only for P8/P9
    if key.endswith("_expanded"):
        cl = [expanded_labels_map.get(lbl, lbl) for lbl in candidate_labels]
        inv_map = {expanded_labels_map.get(lbl, lbl): lbl for lbl in candidate_labels}
    else:
        cl = candidate_labels
        inv_map = None

    preds = clf(
        texts,
        candidate_labels=cl,
        multi_label=False,
        hypothesis_template=template,
    )

    yhat_raw = [obj["labels"][0] for obj in preds]
    # Map expanded labels back to originals for a fair accuracy comparison
    yhat = [inv_map[y] if inv_map else y for y in yhat_raw]

    acc = round((pd.Series(yhat) == pd.Series(true_labels)).mean(), 4)
    ablation_rows.append({"prompt_id": key, "template": template, "accuracy": acc})

result_df = (
    pd.DataFrame(ablation_rows)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

print(f"Best template : {result_df.iloc[0]['template']}")
print(f"Best accuracy : {result_df.iloc[0]['accuracy']}")
result_df

## 5. Notes

- This notebook evaluates classification quality only.
- Summarization quality is evaluated manually in the technical report.
- Increase sample size and run-time if you need tighter confidence intervals.